# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

# Suppress SettingWithCopyWarning (for later usage in pandas)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset '@id': {metadata.id}")
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review all available record sets, their fields, columns, and their `@id`s for exploration.

**Note:** The Croissant schema typically defines logical entities under "record sets" (tables), and each contains fields (columns). We reference each entity by its `@id`.

In [ ]:
# List available record sets and their fields by @id
print("--- Record Sets Overview ---\n")
record_sets = dataset.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - name: {field.name}")
        print(f"      @id: {field.id}")
        if hasattr(field, 'column') and field.column:
            print(f"        column: {field.column.id if hasattr(field.column, 'id') else field.column}")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for further analysis.

Use the record set and field `@id`s printed above. In this dataset, main table is typically called 'Protein Table' or similar, but always reference by its `@id`.

We'll attempt to load all record sets and display the first one.

In [ ]:
# Extract data for each record set, store as DataFrames by @id
dataframes = {}
print("Available RecordSet @ids loaded into DataFrames:")
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"- {rs_id} (rows: {len(df)}, cols: {list(df.columns)})")
    except Exception as e:
        print(f"- {rs_id} FAILED: {e}")

# Preview the first DataFrame
if record_set_ids:
    preview_rs_id = record_set_ids[0]
    print(f"\nColumns in record set '{preview_rs_id}': ", dataframes[preview_rs_id].columns.tolist())
    dataframes[preview_rs_id].head()
else:
    print("No record sets found in the metadata.")

## 4. Exploratory Data Analysis (EDA)

Apply basic EDA: filtering on a numeric field, normalization, grouping by a categorical field.

**Pick the fields by their `@id` as shown above. Adjust the chosen fields based on your data above.**

For demonstration, we'll use likely numeric fields such as `cr:field_coverage_percent` or similar (actual `@id` will appear above).

In [ ]:
# Choose a record set to analyze (use the first one for demo)
rs_id = record_set_ids[0] if record_set_ids else None
# Choose numeric and group fields by @id (replace these if you know the exact field ids from the overview)
df = dataframes[rs_id] if rs_id else pd.DataFrame()

# Try to auto-select a likely numeric field (look for ones with 'percent', 'count', 'MW', 'intensity', etc)
possible_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ['coverage', 'count', 'intens', 'mw', 'abundance', 'peptide', 'score', 'percent', 'sum'])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # Fall back to the first field
    numeric_field = df.columns[0] if not df.empty else None

# Try to auto-select a categorical/group field
possible_group_fields = [col for col in df.columns if any(s in col.lower() for s in ['class', 'type', 'sample', 'group', 'category', 'description'])]
group_field = possible_group_fields[0] if possible_group_fields else None

print(f"Numeric field selected: {numeric_field}")
print(f"Group field selected: {group_field}\n")

if numeric_field is None or df.empty:
    print("No suitable numeric field or data frame for analysis.")
else:
    # Drop NA in selected numeric field
    filtered_df = df.dropna(subset=[numeric_field])
    # Ensure numeric
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    # Filter
    threshold = filtered_df[numeric_field].mean()  # use mean as demo
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):\n", filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by {group_field}:\n", grouped_df.head())

## 5. Visualization

Visualize numeric field distribution (histogram), and (if available) group means as a bar plot.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if not df.empty and numeric_field:
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        # Plot group means
        means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        means.plot(kind='bar', figsize=(8,4))
        plt.title(f"Top 10 Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion

- We loaded the FAIR\^2 dataset via its Croissant schema URL and explored the metadata, record sets, and fields using only their `@id`s.
- We loaded the main tabular data, performed basic EDA (filtering, normalization, grouping), and visualized data distributions.
- For further analysis, consult the Croissant schema's `@id` fields for clear, reproducible column references and study the original publication with this dataset for semantic context.